# Gate-2 pooled-vector identity check (CPU)

Confirms the pooled vectors co-extracted with the tokens reproduce the separately
frozen pooled vectors the pooled pipeline uses -- proving the token arm and the
pooled arm share one representation (the primary causal pair depends on this).

**No GPU.** Enable Internet + the `GITHUB_TOKEN` secret (to import the tested
comparator). Attach any of:
- the **EEG token run** dataset (`token_index.json`) and the **P4b EEG vectors**
  dataset (`vector_index.csv`) -> EEG identity check;
- the **text token run** dataset (`text_token_index.csv`) and the **frozen text
  vectors** dataset (`text_vector_index.csv`) -> text identity check.
Each check runs only if both its inputs are attached.

In [ ]:
import csv, glob, hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
import numpy as np
from kaggle_secrets import UserSecretsClient
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = '0508328aebc574437b7016cb216fda1f83b72591'
WORKTREE = '/kaggle/working/SemKey'
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as h:
    h.write("#!/usr/bin/env python3\nimport os, sys\np = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in p else 'x-access-token')\n")
os.chmod(askpass, 0o700)
cenv = os.environ.copy(); cenv.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=cenv)
finally:
    os.remove(askpass); del github_token, cenv
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
sys.path.insert(0, WORKTREE)
from evaluation.pooled_vector_identity import compare_pooled_vectors
print('comparator imported at', COMMIT[:10])

In [ ]:
def find_dirs(filename):
    return [Path(os.path.dirname(h)) for h in glob.glob('/kaggle/input/**/' + filename, recursive=True)]

def one(dirs, what):
    if len(dirs) > 1:
        raise AssertionError(('attached more than one ' + what, [str(d) for d in dirs]))
    return dirs[0] if dirs else None

def pick_text_token_dir():
    # A stale text-token copy may be bundled inside the EEG token dataset. Prefer the
    # STANDALONE text-token dir (one with no EEG token_index.json beside it), so the
    # freshly re-extracted dataset wins even if the old bundled copy is still attached.
    dirs = find_dirs('text_token_index.csv')
    if len(dirs) <= 1:
        return dirs[0] if dirs else None
    standalone = [d for d in dirs if not glob.glob(str(d.parent) + '/**/token_index.json', recursive=True)]
    if len(standalone) == 1:
        return standalone[0]
    raise AssertionError(('cannot disambiguate text_token_index.csv',
                          [str(d) for d in dirs], 'standalone', [str(d) for d in standalone]))

def text_provenance(root):
    if root is None:
        return None
    man = json.load(open(root / 'text_token_manifest.json', encoding='utf-8'))
    prov = {'combined_chunk_sha256': man.get('combined_chunk_sha256')}
    rm = root / 'run_metadata.json'
    if rm.exists():
        prov['project_commit'] = json.load(open(rm, encoding='utf-8')).get('project_commit')
    return prov

eeg_token_root = one(find_dirs('token_index.json'), 'token_index.json')
p4b_vec_root = one(find_dirs('vector_index.csv'), 'vector_index.csv')
text_token_root = pick_text_token_dir()
text_vec_root = one(find_dirs('text_vector_index.csv'), 'text_vector_index.csv')
print({'eeg_token_run': str(eeg_token_root), 'p4b_eeg_vectors': str(p4b_vec_root),
       'text_token_run': str(text_token_root), 'frozen_text_vectors': str(text_vec_root)})
print('TEXT TOKEN PROVENANCE (must be the fp16 re-extraction):', text_provenance(text_token_root))

def _load_npz_col(root, rel, col='vectors'):
    with np.load(root / rel) as a:
        return a[col]

def load_eeg_token_run(root):
    idx = json.load(open(root / 'token_index.json', encoding='utf-8'))
    ids, vecs = [], []
    for entry in idx['chunks']:
        meta = json.load(open(root / (entry['token_file'][:-4] + '.json'), encoding='utf-8'))
        v = _load_npz_col(root, entry['token_file'])
        assert v.shape[0] == len(meta['trial_ids'])
        ids += list(meta['trial_ids']); vecs.append(v)
    return ids, np.concatenate(vecs, axis=0)

def load_indexed_vectors(root, index_csv, id_field, file_field, offset_field, filter_fn=None, assert_prompt=None):
    rows = list(csv.DictReader(open(root / index_csv, encoding='utf-8-sig')))
    if filter_fn is not None:
        rows = [r for r in rows if filter_fn(r)]
    assert rows, ('no rows after filter in ' + index_csv)
    if assert_prompt is not None:
        modes = sorted({r['prompt_mode'] for r in rows})
        assert modes == [assert_prompt], ('prompt_mode', modes)
    ids, vecs, cache = [], [], {}
    for r in rows:
        f = r[file_field]
        if f not in cache:
            cache[f] = _load_npz_col(root, f)
        ids.append(r[id_field]); vecs.append(cache[f][int(r[offset_field])])
    return ids, np.stack(vecs)

In [ ]:
eeg_report = None
if eeg_token_root and p4b_vec_root:
    tok_ids, tok_vecs = load_eeg_token_run(eeg_token_root)
    p4b_ids, p4b_vecs = load_indexed_vectors(
        p4b_vec_root, 'vector_index.csv', 'target_trial_id', 'vector_file', 'vector_offset',
        filter_fn=lambda r: r['condition'] in ('correct_train', 'correct_val'),
        assert_prompt='all_masked')
    eeg_report = compare_pooled_vectors(tok_ids, tok_vecs, p4b_ids, p4b_vecs)
    print('EEG pooled-vector identity:', json.dumps(eeg_report, indent=2, sort_keys=True))
    print('EEG MATCH:', eeg_report['match'])
else:
    print('EEG check skipped (need both the EEG token run and P4b EEG vectors attached)')

In [ ]:
text_report = None
if text_token_root and text_vec_root:
    ttok_ids, ttok_vecs = load_indexed_vectors(
        text_token_root, 'text_token_index.csv', 'text_target_id', 'token_file', 'token_offset')
    tvec_ids, tvec_vecs = load_indexed_vectors(
        text_vec_root, 'text_vector_index.csv', 'text_target_id', 'vector_file', 'vector_offset')
    # fp16 bar: the text encoder runs T5 in fp16 autocast (the EEG path is fp32), so
    # ~1e-3 cosine drift between independent runs is the reproducibility floor, not an
    # error. Any REAL mismatch (stale data, task-prompted vs all_masked, misalignment)
    # collapses mean cosine far below this -- the pre-fix run sat at 0.795 / -0.37.
    text_report = compare_pooled_vectors(ttok_ids, ttok_vecs, tvec_ids, tvec_vecs,
                                         atol=0.1, rtol=0.01, min_cosine=0.99)
    print('Text pooled-vector identity:', json.dumps(text_report, indent=2, sort_keys=True))
    print('TEXT MATCH:', text_report['match'])
else:
    print('Text check skipped (need both the text token run and frozen text vectors attached)')

In [ ]:
report = {'project_commit': COMMIT, 'eeg_identity': eeg_report, 'text_identity': text_report}
out = '/kaggle/working/gate2_identity_report.json'
with open(out, 'w', encoding='utf-8') as h:
    json.dump(report, h, indent=2, sort_keys=True); h.write('\n')
ran = [k for k, v in (('eeg', eeg_report), ('text', text_report)) if v is not None]
matches = [v['match'] for v in (eeg_report, text_report) if v is not None]
ok = bool(matches) and all(matches)
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
print('checks run:', ran)
print('ALL RUN CHECKS MATCH:', ok)
print('wrote', out)

A `match: true` on each attached check confirms the token arm's pooled vector is
the same representation the frozen pooled pipeline uses (within float16
tolerance). Record the report. If either match is **false**, do NOT proceed to
freeze -- inspect: a systematic offset usually means a provenance mismatch (e.g.
the frozen vectors were task-prompted, not `all_masked`). Held-out test stays
sealed.